# Lab 1: API smoke-test моделей

Ноутбук является первичным протоколом эксперимента: метрики и полные ответы перенесены из четырёх исходных логов от 18.09.2026. Ниже сохранены имена исходных логов, конфигурация, значения `api_usage` и ответы без содержательных правок. Сетевых вызовов и секретов здесь нет.

In [ ]:
run_config = {
    'date': '2026-09-18',
    'base_url': 'https://openrouter.ai/api/v1',
    'api': 'OpenAI-compatible Chat Completions',
    'stream': True,
    'max_tokens': 1199,
    'temperature': 'provider default',
    'prompt_version': 'lab1-smoke-v1',
    'measurement_script': 'llm_metrics.py',
}
run_config

In [ ]:
runs = [
    {
        'model': 'openai/gpt-5.6-sol',
        'source_log': 'gpt-5_6-sol.txt',
        'finish_reason': 'stop',
        'total_time_s': 5.427549,
        'ttft_event_s': 4.102754,
        'ttft_content_s': 4.638078,
        'time_after_first_token_s': 0.789471,
        'prompt_tokens': 214,
        'completion_tokens': 252,
        'total_tokens': 466,
        'token_source': 'api_usage',
        'content_chunks': 143,
        'tokens_per_second': 46.430,
        'input_usd_per_million': 4.00,
        'output_usd_per_million': 20.00,
        'format': 'raw JSON; 10 expected fields',
    },
    {
        'model': 'mistralai/mistral-medium-3-5',
        'source_log': 'mistral-medium-3-5.txt',
        'finish_reason': 'stop',
        'total_time_s': 1.338671,
        'ttft_event_s': 0.894906,
        'ttft_content_s': 0.894906,
        'time_after_first_token_s': 0.443765,
        'prompt_tokens': 274,
        'completion_tokens': 76,
        'total_tokens': 350,
        'token_source': 'api_usage',
        'content_chunks': 28,
        'tokens_per_second': 56.773,
        'input_usd_per_million': 1.50,
        'output_usd_per_million': 7.50,
        'format': 'plain text; not JSON',
    },
    {
        'model': 'qwen/qwen3.8-max',
        'source_log': 'qwen3_8-max.txt',
        'finish_reason': 'stop',
        'total_time_s': 55.148368,
        'ttft_event_s': 1.781531,
        'ttft_content_s': 52.433001,
        'time_after_first_token_s': 2.715367,
        'prompt_tokens': 283,
        'completion_tokens': 2136,
        'total_tokens': 2419,
        'token_source': 'api_usage',
        'content_chunks': 43,
        'tokens_per_second': 38.732,
        'input_usd_per_million': 2.00,
        'output_usd_per_million': 6.00,
        'format': 'raw JSON; 10 expected fields',
    },
    {
        'model': 'deepseek/deepseek-v4-pro',
        'source_log': 'deepseek-v4-pro.txt',
        'finish_reason': 'stop',
        'total_time_s': 27.688746,
        'ttft_event_s': 1.470071,
        'ttft_content_s': 25.108124,
        'time_after_first_token_s': 2.580622,
        'prompt_tokens': 243,
        'completion_tokens': 1540,
        'total_tokens': 1783,
        'token_source': 'api_usage',
        'content_chunks': 49,
        'tokens_per_second': 55.618,
        'input_usd_per_million': 1.32,
        'output_usd_per_million': 3.96,
        'format': 'JSON in Markdown; incompatible keys',
    },
]

for run in runs:
    assert run['prompt_tokens'] + run['completion_tokens'] == run['total_tokens']

len(runs)

In [ ]:
raw_completions = {
    'openai/gpt-5.6-sol': '''{
  "category": "unknown",
  "urgency": "unknown",
  "summary": "Текст обращения, история, фрагменты базы знаний и JSON Schema не предоставлены.",
  "known_data": [],
  "missing_data": [
    "Текст обращения пользователя",
    "История обращения",
    "Фрагменты базы знаний",
    "Требуемая JSON Schema"
  ],
  "relevant_sources": [],
  "next_step": "clarify",
  "draft_response": "Пожалуйста, предоставьте текст обращения, историю, фрагменты базы знаний и JSON Schema для результата.",
  "route": "manual_review",
  "operator_required": true
}''',
    'mistralai/mistral-medium-3-5': '''Пожалуйста, предоставьте:
1. Текст обращения пользователя
2. Историю обращений (если есть)
3. Фрагменты базы знаний (если есть)

После этого я смогу проанализировать информацию и вернуть результат в формате JSON в соответствии с вашими требованиями.''',
    'qwen/qwen3.8-max': '''{
  "category": "unknown",
  "urgency": "unknown",
  "summary": "Не переданы текст обращения, история обращения и фрагменты базы знаний, поэтому первичный разбор невозможен.",
  "known_data": [],
  "missing_data": [
    "текст обращения пользователя",
    "история обращения",
    "фрагменты базы знаний",
    "контекст или идентификатор обращения"
  ],
  "relevant_sources": [],
  "next_step": "clarify",
  "draft_response": "Для первичного разбора предоставьте, пожалуйста, текст обращения пользователя, историю обращения и подходящие фрагменты базы знаний.",
  "route": "clarification",
  "operator_required": false
}''',
    'deepseek/deepseek-v4-pro': '''```json
{
  "категория_обращения": null,
  "срочность": null,
  "краткое_содержание": null,
  "известные_данные": {},
  "недостающие_данные": [
    "текст обращения пользователя",
    "история обращений",
    "фрагменты базы знаний",
    "JSON Schema для возврата"
  ],
  "релевантные_источники": [],
  "следующий_шаг": "clarify",
  "черновик_ответа_пользователю": "Недостаточно данных для анализа обращения. Пожалуйста, предоставьте текст обращения, историю и фрагменты базы знаний.",
  "маршрут_обращения": "support_clarification",
  "участие_оператора": true
}
```''',
}

assert set(raw_completions) == {run['model'] for run in runs}
for model, completion in raw_completions.items():
    print(f'--- {model} ---')
    print(completion)

In [ ]:
import json

expected_fields = {
    'category', 'urgency', 'summary', 'known_data', 'missing_data',
    'relevant_sources', 'next_step', 'draft_response', 'route',
    'operator_required',
}
format_checks = {}
for model, completion in raw_completions.items():
    fenced = completion.startswith('```json\n') and completion.endswith('\n```')
    candidate = completion[8:-4] if fenced else completion
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        parsed = None
    format_checks[model] = {
        'raw_json': isinstance(parsed, dict) and not fenced,
        'markdown_fenced_json': isinstance(parsed, dict) and fenced,
        'has_exact_expected_fields': (
            isinstance(parsed, dict) and set(parsed) == expected_fields
        ),
    }

format_checks

In [ ]:
for run in runs:
    run['estimated_cost_usd'] = (
        run['prompt_tokens'] * run['input_usd_per_million']
        + run['completion_tokens'] * run['output_usd_per_million']
    ) / 1_000_000

columns = [
    'model', 'total_time_s', 'ttft_content_s', 'prompt_tokens',
    'completion_tokens', 'total_tokens', 'estimated_cost_usd', 'format'
]
print('| ' + ' | '.join(columns) + ' |')
print('| ' + ' | '.join(['---'] * len(columns)) + ' |')
for run in runs:
    values = [
        run['model'],
        f"{run['total_time_s']:.6f}",
        f"{run['ttft_content_s']:.6f}",
        str(run['prompt_tokens']),
        str(run['completion_tokens']),
        str(run['total_tokens']),
        f"{run['estimated_cost_usd']:.6f}",
        run['format'],
    ]
    print('| ' + ' | '.join(values) + ' |')

In [ ]:
by_latency = sorted(runs, key=lambda row: row['total_time_s'])
by_tokens = sorted(runs, key=lambda row: row['total_tokens'])
by_cost = sorted(runs, key=lambda row: row['estimated_cost_usd'])

print('Fastest:', by_latency[0]['model'])
print('Fewest tokens:', by_tokens[0]['model'])
print('Lowest estimated cost:', by_cost[0]['model'])
print('GPT/Qwen token ratio:', round(runs[2]['total_tokens'] / runs[0]['total_tokens'], 2))
print('GPT/Qwen latency ratio:', round(runs[2]['total_time_s'] / runs[0]['total_time_s'], 2))

## Проектный токен-бюджет

Это внутренние верхние лимиты команды, а не опубликованные характеристики моделей и не средние значения датасета. Лимит инструкции сверяется с фактическим диапазоном `api_usage` из smoke-теста; остальные значения задают допустимый размер компонентов перед отправкой запроса.

In [ ]:
observed_instruction_input_range = (
    min(run['prompt_tokens'] for run in runs),
    max(run['prompt_tokens'] for run in runs),
)
token_budget = {
    'instruction_and_schema': {
        'tokens': 450,
        'basis': (
            f'observed instruction-only api_usage range '
            f'{observed_instruction_input_range[0]}-'
            f'{observed_instruction_input_range[1]}; reserve for JSON Schema'
        ),
    },
    'ticket': {
        'tokens': 250,
        'basis': 'team design limit for one incoming ticket',
    },
    'history': {
        'tokens': 600,
        'basis': 'team design limit after relevance filtering and truncation',
    },
    'retrieved_kb_chunks': {
        'tokens': 3 * 300,
        'basis': 'at most 3 retrieved chunks x approximately 300 tokens',
    },
    'structured_result': {
        'tokens': 500,
        'basis': 'target visible JSON limit; GPT smoke run used 252 tokens',
    },
}

single_call_budget = sum(item['tokens'] for item in token_budget.values())
one_retry_budget = single_call_budget * 2
assert single_call_budget == 2700
assert one_retry_budget == 5400

for component, item in token_budget.items():
    print(f"{component}: {item['tokens']} — {item['basis']}")
print('single call:', single_call_budget)
print('one full retry:', one_retry_budget)

## Вывод

Для следующего эксперимента выбран `openai/gpt-5.6-sol`: он вернул машинно-читаемый JSON при умеренных задержке и расходе. `mistralai/mistral-medium-3-5` остаётся резервом и должен быть повторно проверен с реальным `response_format: json_schema`.

Этот smoke-тест не измеряет качество разбора: в запросе не было обращения, истории, фрагментов БЗ и JSON Schema. Для оценки качества нужны 10 golden-примеров, не менее трёх повторов и автоматическая валидация схемы.